In [2]:
!nvidia-smi

Wed Aug 12 17:47:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd ~
!git clone https://github.com/sahilahmed21/Atlas.git Atlas
%cd ~/Atlas

/root
fatal: destination path 'Atlas' already exists and is not an empty directory.
/root/Atlas


In [4]:
!pip install -U pip
!pip install uv

In [5]:
!uv sync

Resolved 111 packages in 1ms
Checked 108 packages in 2ms


In [6]:
!uv run python --version

Python 3.11.15


In [7]:
!pip install "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 MB 34.7 MB/s  0:00:09


In [8]:
!python scripts/verify_wsl_vllm.py

vllm 0.26.0 (pin ok) 0.26.0
torch 2.11.0+cu128
cuda True
device Tesla T4


In [9]:
!python -c "import vllm; print(vllm.__version__)"

0.26.0


In [10]:
%cd ~/Atlas

!CUDA_VISIBLE_DEVICES=0 nohup uv run vllm serve Qwen/Qwen2.5-0.5B-Instruct \
  --port 8001 \
  --gpu-memory-utilization 0.4 \
  --max-model-len 2048 \
  > /tmp/vllm-8001.log 2>&1 &

/root/Atlas


In [31]:
!cat /tmp/vllm-8001.log

(APIServer pid=3213) INFO 08-12 17:50:49 [api_utils.py:345] 
(APIServer pid=3213) INFO 08-12 17:50:49 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=3213) INFO 08-12 17:50:49 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.26.0
(APIServer pid=3213) INFO 08-12 17:50:49 [api_utils.py:345]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=3213) INFO 08-12 17:50:49 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=3213) INFO 08-12 17:50:49 [api_utils.py:345] 
(APIServer pid=3213) INFO 08-12 17:50:49 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'port': 8001, 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'max_model_len': 2048, 'gpu_memory_utilization': 0.4}
(APIServer pid=3213) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=3213) INFO 08-12 17:50:50 [model.py:623] Resolved architecture: Qwen2ForCau

In [38]:
!curl -s http://127.0.0.1:8001/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1786557210,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-97a901a62c0e0b67","object":"model_permission","created":1786557210,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [46]:
%cd ~/Atlas

!CUDA_VISIBLE_DEVICES=0 nohup uv run vllm serve Qwen/Qwen2.5-0.5B-Instruct \
  --port 8002 \
  --gpu-memory-utilization 0.4 \
  --max-model-len 2048 \
  > /tmp/vllm-8002.log 2>&1 &

/root/Atlas


In [52]:
!ss -ltnp | grep -E ':8001|:8002'

LISTEN 0      2048         0.0.0.0:8001       0.0.0.0:*    users:(("vllm",pid=2131,fd=27))           
LISTEN 0      2048         0.0.0.0:8002       0.0.0.0:*    users:(("vllm",pid=2143,fd=27))           


In [50]:
!cat /tmp/vllm-8002.log

(APIServer pid=5432) INFO 08-12 17:55:01 [api_utils.py:345] 
(APIServer pid=5432) INFO 08-12 17:55:01 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=5432) INFO 08-12 17:55:01 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.26.0
(APIServer pid=5432) INFO 08-12 17:55:01 [api_utils.py:345]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=5432) INFO 08-12 17:55:01 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=5432) INFO 08-12 17:55:01 [api_utils.py:345] 
(APIServer pid=5432) INFO 08-12 17:55:01 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'port': 8002, 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'max_model_len': 2048, 'gpu_memory_utilization': 0.4}
(APIServer pid=5432) Traceback (most recent call last):
(APIServer pid=5432)   File "/usr/local/bin/vllm", line 6, in <module>
(APIServer pid=5432)     sys.exit(main())
(APIServer pid=5432)              ^^^^^^
(APIServer pid=5432)   File "/usr/local

In [53]:
!curl -s http://127.0.0.1:8002/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1786557426,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-87b03c795e7c2bbe","object":"model_permission","created":1786557426,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [54]:
%cd ~/Atlas
!uv run python benchmarks/run_routing_matrix.py --help | grep worker-mode

/root/Atlas
/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
usage: run_routing_matrix.py [-h] [--worker-mode {simulated,live}]
  --worker-mode {simulated,live}


In [42]:
import requests

payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [{"role": "user", "content": "Say ping"}],
    "max_tokens": 10,
    "stream": False,
}

r = requests.post(
    "http://127.0.0.1:8001/v1/chat/completions",
    json=payload,
    timeout=60,
)

print(r.status_code)
print(r.text)

200
{"id":"chatcmpl-892f196d3e2e766f","object":"chat.completion","created":1786557240,"model":"Qwen/Qwen2.5-0.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"Hello! How can I assist you today?","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.26.0-4f1bb46d","usage":{"prompt_tokens":31,"total_tokens":41,"completion_tokens":10,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}


In [43]:
import requests

payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [{"role": "user", "content": "Say ping"}],
    "max_tokens": 10,
    "stream": False,
}

r = requests.post(
    "http://127.0.0.1:8002/v1/chat/completions",
    json=payload,
    timeout=60,
)

print(r.status_code)
print(r.text)

200
{"id":"chatcmpl-98e736e5d3bceac2","object":"chat.completion","created":1786557247,"model":"Qwen/Qwen2.5-0.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"Hello! How can I assist you today?","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.26.0-4f1bb46d","usage":{"prompt_tokens":31,"total_tokens":41,"completion_tokens":10,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}


In [44]:
%cd ~/Atlas

/root/Atlas


In [45]:
!uv run python benchmarks/run_routing_matrix.py \
  --worker-mode live \
  --patterns high_reuse \
  --strategies round_robin,prefix_aware \
  --n 24 \
  --hardware colab-t4 \
  --vllm-version 0.26.0 \
  --replica-mode time_sliced_dual \
  --worker-a-url http://127.0.0.1:8001/v1 \
  --worker-b-url http://127.0.0.1:8002/v1

/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
Failed to detach context
Traceback (most recent call last):
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 154, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/contextvars_context.py", line 50, in detach
    self._current_context.reset(token)  # type: ignore
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x7f803d66a1b0> at 0x7f803ca4cd00> was created in a different Context
Failed to detach context
Traceback (most recent call last):
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 154, in detach
    _RUNTIME_C

In [ ]:
!ls -lh results/phase5-live/

In [ ]:
!uv run python benchmarks/run_routing_matrix.py \
  --worker-mode live \
  --patterns high_reuse \
  --strategies round_robin,prefix_aware \
  --n 24 \
  --hardware colab-t4 \
  --vllm-version 0.26.0 \
  --replica-mode time_sliced_dual \
  --worker-a-url http://127.0.0.1:8001/v1 \
  --worker-b-url http://127.0.0.1:8002/v1

In [ ]:
!ls -lah /root/Atlas/results/phase5-live

In [ ]:
import pandas as pd

df = pd.read_csv("/root/Atlas/results/phase5/routing_matrix.csv")
display(df)

In [ ]:
print(df.columns.tolist())
print(df.to_string(index=False))

In [ ]:
print("Rows:", len(df))
print("Patterns:", df["pattern"].unique())
print("Strategies:", df["strategy"].unique())

In [ ]:
!uv run python benchmarks/run_routing_matrix.py --help

In [ ]:
!grep -n "worker-mode\|patterns\|strategies\|phase5-live\|routing_matrix" benchmarks/run_routing_matrix.py

In [55]:
%cd ~/Atlas

!git fetch origin
!git checkout master
!git pull --ff-only

!git log -1 --oneline

/root/Atlas
Already on 'master'
Your branch is up to date with 'origin/master'.
Already up to date.
b4dcce6 (HEAD -> master, origin/master, origin/HEAD) docs: Phase 7 Colab infra PASS; sim matrix is not live evidence


In [56]:
!uv run python benchmarks/run_routing_matrix.py --help | grep worker-mode

/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
usage: run_routing_matrix.py [-h] [--worker-mode {simulated,live}]
  --worker-mode {simulated,live}


In [57]:
%cd ~/Atlas

!uv run python benchmarks/run_routing_matrix.py \
  --worker-mode live \
  --patterns high_reuse \
  --strategies round_robin,prefix_aware \
  --n 24 \
  --hardware colab-t4 \
  --vllm-version 0.26.0 \
  --replica-mode time_sliced_dual \
  --worker-a-url http://127.0.0.1:8001/v1 \
  --worker-b-url http://127.0.0.1:8002/v1

/root/Atlas
/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
Failed to detach context
Traceback (most recent call last):
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 154, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/contextvars_context.py", line 50, in detach
    self._current_context.reset(token)  # type: ignore
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x7d3f4c55da80> at 0x7d3f4a8d6080> was created in a different Context
Failed to detach context
Traceback (most recent call last):
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 154, in detach
  

In [58]:
from google.colab import files

files.download(
    "/root/Atlas/results/phase5-live/routing_matrix_live.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
!git fetch origin
!git checkout master
!git pull --ff-only

remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 49 (delta 28), reused 49 (delta 28), pack-reused 0 (from 0)
Unpacking objects: 100% (49/49), 16.74 KiB | 1.12 MiB/s, done.
From https://github.com/sahilahmed21/Atlas
   b4dcce6..253f935  master     -> origin/master
Already on 'master'
Your branch is behind 'origin/master' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating b4dcce6..253f935
error: The following untracked working tree files would be overwritten by merge:
	results/phase5-live/routing_matrix_live.csv
Please move or remove them before you merge.
Aborting


In [62]:
!git log -1 --oneline

b4dcce6 (HEAD -> master) docs: Phase 7 Colab infra PASS; sim matrix is not live evidence


In [63]:
!uv run python benchmarks/run_routing_matrix.py --help | grep phase8

/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [64]:
!uv run python benchmarks/run_routing_matrix.py --help | grep worker-mode


/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
usage: run_routing_matrix.py [-h] [--worker-mode {simulated,live}]
  --worker-mode {simulated,live}


In [65]:
!curl -s http://127.0.0.1:8001/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1786559977,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-98bb5e21e7bb1281","object":"model_permission","created":1786559977,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [66]:
!curl -s http://127.0.0.1:8002/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1786559987,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-8c6a6d95d7720b73","object":"model_permission","created":1786559987,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [67]:
!uv run python benchmarks/run_routing_matrix.py \
  --phase8 \
  --worker-mode live \
  --n 24 \
  --hardware colab-t4 \
  --vllm-version 0.26.0 \
  --replica-mode time_sliced_dual \
  --worker-a-url http://127.0.0.1:8001/v1 \
  --worker-b-url http://127.0.0.1:8002/v1 \
  --out results/phase8/gate_matrix_live.csv

/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
usage: run_routing_matrix.py [-h] [--worker-mode {simulated,live}]
                             [--patterns PATTERNS] [--strategies STRATEGIES]
                             [--n N] [--out OUT] [--hardware HARDWARE]
                             [--vllm-version VLLM_VERSION]
                             [--replica-mode REPLICA_MODE]
                             [--worker-a-url WORKER_A_URL]
                             [--worker-b-url WORKER_B_URL]
run_routing_matrix.py: error: unrecognized arguments: --phase8


In [71]:
%cd ~/Atlas

!git fetch origin
!git checkout master
!git reset --hard origin/master
!git log -1 --oneline

/root/Atlas
Already on 'master'
Your branch is behind 'origin/master' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 253f935 feat: Phase 8 prefix load gate + sim before/after matrix (GREEN)
253f935 (HEAD -> master, origin/master, origin/HEAD) feat: Phase 8 prefix load gate + sim before/after matrix (GREEN)


In [72]:
!git show --stat --oneline HEAD

253f935 (HEAD -> master, origin/master, origin/HEAD) feat: Phase 8 prefix load gate + sim before/after matrix (GREEN)
 .gitignore                                     |   1 +
 benchmarks/README.md                           |  13 ++-
 benchmarks/run_routing_matrix.py               |  95 ++++++++++++-----
 benchmarks/test_routing_matrix_harness.py      |  24 +++++
 configs/routing/strategies.yaml                |   6 +-
 docs/HANDOFF.md                                | 138 ++++++++++++-------------
 docs/phases/phase-6/CLAIM_INVENTORY.md         |   6 +-
 docs/phases/phase-8/ACCEPTANCE.md              |  67 ++----------
 docs/phases/phase-8/README.md                  |  64 ++++--------
 docs/testing/phase-8.tdd.md                    |  36 +++++++
 platform/gateway/app.py                        |  36 ++++++-
 platform/gateway/test_gateway_routing_state.py |  27 ++++-
 platform/router/strategies.py                  |  23 ++++-
 platform/router/test_router_strategies.py      |  71 ++++++++++

In [76]:
%cd ~/Atlas

!git fetch origin
!git checkout master
!git reset --hard origin/master

print("=== COMMIT ===")
!git log -1 --oneline

print("=== CLI ===")
!uv run python benchmarks/run_routing_matrix.py --help

print("=== PHASE8 FILES ===")
!find results -maxdepth 2 -type f | sort | tail -50

print("=== PHASE8 CODE ===")
!grep -Rni "load_margin\|hit_broken\|PrefixAwareRouter\|phase8" platform workers benchmarks | head -100

/root/Atlas
Already on 'master'
Your branch is up to date with 'origin/master'.
HEAD is now at 253f935 feat: Phase 8 prefix load gate + sim before/after matrix (GREEN)
=== COMMIT ===
253f935 (HEAD -> master, origin/master, origin/HEAD) feat: Phase 8 prefix load gate + sim before/after matrix (GREEN)
=== CLI ===
/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
usage: run_routing_matrix.py [-h] [--worker-mode {simulated,live}]
                             [--patterns PATTERNS] [--strategies STRATEGIES]
                             [--n N] [--out OUT] [--hardware HARDWARE]
                             [--vllm-version VLLM_VERSION]
                             [--replica-mode REPLICA_MODE]
                             [--worker-a-url WORKER_A_URL]
                             [--worker-b-ur

In [77]:
%cd ~/Atlas

!uv run python benchmarks/run_routing_matrix.py \
  --phase8 \
  --worker-mode live \
  --n 24 \
  --hardware colab-t4 \
  --vllm-version 0.26.0 \
  --replica-mode time_sliced_dual \
  --worker-a-url http://127.0.0.1:8001/v1 \
  --worker-b-url http://127.0.0.1:8002/v1 \
  --out results/phase8/gate_matrix_live.csv

/root/Atlas
/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
Failed to detach context
Traceback (most recent call last):
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 154, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/contextvars_context.py", line 50, in detach
    self._current_context.reset(token)  # type: ignore
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: <Token var=<ContextVar name='current_context' default={} at 0x7fab3f191030> at 0x7fab3e564300> was created in a different Context
Failed to detach context
Traceback (most recent call last):
  File "/root/Atlas/.venv/lib/python3.11/site-packages/opentelemetry/context/__init__.py", line 154, in detach
  

In [78]:
from google.colab import files

files.download(
    "/root/Atlas/results/phase8/gate_matrix_live.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [79]:
import pandas as pd

df = pd.read_csv("/root/Atlas/results/phase8/gate_matrix_live.csv")
display(df)

,pattern,strategy,n,ttft_p50_ms,ttft_p95_ms,tokens_per_s_mean,cache_hit_pct,hit_broken_pct,worker_skew,worker_counts,worker_mode,load_margin,hardware,vllm_version,replica_mode,model
0,high_reuse,round_robin,24,35.857,44.626,0.0,0.00,0.0,0.5,"{'worker-a': 12, 'worker-b': 12}",live,0,colab-t4,0.26.0,time_sliced_dual,Qwen/Qwen2.5-0.5B-Instruct
1,high_reuse,prefix_aware,24,26.441,32.300,0.0,95.83,0.0,1.0,{'worker-a': 24},live,0,colab-t4,0.26.0,time_sliced_dual,Qwen/Qwen2.5-0.5B-Instruct
2,high_reuse,prefix_aware,24,33.174,49.017,0.0,45.83,50.0,0.5,"{'worker-a': 12, 'worker-b': 12}",live,1,colab-t4,0.26.0,time_sliced_dual,Qwen/Qwen2.5-0.5B-Instruct
